# 搓一个土的掉渣的Agent

In [1]:
import os
from dotenv import load_dotenv

load_dotenv("apikey.env")

True

In [11]:
DEEPSEEK_API = os.getenv("DEEPSEEK-API-KEY")
base_url = 'https://api.deepseek.com'
if DEEPSEEK_API:
    print("sucessful!")
else:
    print("nedd API KEY")
chat_model = "deepseek-chat"

sucessful!


In [4]:
from openai import OpenAI
client = OpenAI(
    api_key = DEEPSEEK_API,
    base_url = base_url
)

In [12]:
def get_completion(prompt):
    response = client.chat.completions.create(
        model=chat_model,  # 填写需要调用的模型名称
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

In [9]:
response = get_completion("我叫一声你敢答应吗，你是谁？")
print(response)

哈哈，这句经典台词我可太熟悉了！🤣 我是DeepSeek，一个由深度求索公司创造的AI助手。 

虽然我不会像金角大王的葫芦那样被一声呼唤就收走，但我很乐意回应你的召唤！😄 我可以帮你解答问题、聊天、处理文档，或者协助你完成各种任务。

既然你都“叫”了我一声，那我当然要答应啦！有什么需要我帮你的吗？无论是学习、工作还是日常生活中的问题，我都很乐意助你一臂之力！✨


## LangGPT 中的 Role （角色）模板
**Role: Your_Role_Name**

**Profile**
- Author: YZFly
- Version: 0.1
- Language: English or 中文 or Other language
- Description: Describe your role. Give an overview of the character's characteristics and skills

**Skill-1**
- 1.技能描述1 2.技能描述2

**Skill-2**
- 1.技能描述1 2.技能描述2

**Rules**
- Don't break character under any circumstance.
- Don't talk nonsense and make up facts.

**Workflow**
- First, xxx
- Then, xxx
- Finally, xxx

**Initialization**
`As a/an < Role >, you must follow the < Rules >, you must talk to user in default < Language >，you must greet the user. Then introduce yourself and introduce the < Workflow >.`

Prompt Chain 将原有需求分解，通过用多个小的 Prompt 来串联/并联，共同解决一项复杂任务。

In [ ]:
sys_prompt = """你是一个聪明的客服。您将能够根据用户的问题将不同的任务分配给不同的人。您有以下业务线：
1.用户注册。如果用户想要执行这样的操作，您应该发送一个带有"registered workers"的特殊令牌。并告诉用户您正在调用它。
2.用户数据查询。如果用户想要执行这样的操作，您应该发送一个带有"query workers"的特殊令牌。并告诉用户您正在调用它。
3.删除用户数据。如果用户想执行这种类型的操作，您应该发送一个带有"delete workers"的特殊令牌。并告诉用户您正在调用它。
"""
registered_prompt = """
您的任务是根据用户信息存储数据。您需要从用户那里获得以下信息：
1.用户名、性别、年龄
2.用户设置的密码
3.用户的电子邮件地址
如果用户没有提供此信息，您需要提示用户提供。如果用户提供了此信息，则需要将此信息存储在数据库中，并告诉用户注册成功。
存储方法是使用SQL语句。您可以使用SQL编写插入语句，并且需要生成用户ID并将其返回给用户。
如果用户没有新问题，您应该回复带有 "customer service" 的特殊令牌，以结束任务。
"""
query_prompt = """
您的任务是查询用户信息。您需要从用户那里获得以下信息：
1.用户ID
2.用户设置的密码
如果用户没有提供此信息，则需要提示用户提供。如果用户提供了此信息，那么需要查询数据库。如果用户ID和密码匹配，则需要返回用户的信息。
如果用户没有新问题，您应该回复带有 "customer service" 的特殊令牌，以结束任务。
"""
delete_prompt = """
您的任务是删除用户信息。您需要从用户那里获得以下信息：
1.用户ID
2.用户设置的密码
3.用户的电子邮件地址
如果用户没有提供此信息，则需要提示用户提供该信息。
如果用户提供了这些信息，则需要查询数据库。如果用户ID和密码匹配，您需要通知用户验证码已发送到他们的电子邮件，需要进行验证。
如果用户没有新问题，您应该回复带有 "customer service" 的特殊令牌，以结束任务。
"""


In [20]:
class SmartAssistant:
    def __init__(self):
        self.client = client 

        self.system_prompt = sys_prompt
        self.registered_prompt = registered_prompt
        self.query_prompt = query_prompt
        self.delete_prompt = delete_prompt

        # Using a dictionary to store different sets of messages
        self.messages = {
            "system": [{"role": "system", "content": self.system_prompt}],
            "registered": [{"role": "system", "content": self.registered_prompt}],
            "query": [{"role": "system", "content": self.query_prompt}],
            "delete": [{"role": "system", "content": self.delete_prompt}]
        }

        # Current assignment for handling messages
        self.current_assignment = "system"

    def get_response(self, user_input):
        self.messages[self.current_assignment].append({"role": "user", "content": user_input})
        while True:
            response = self.client.chat.completions.create(
                model=chat_model,
                messages=self.messages[self.current_assignment],
                temperature=0.9,
                stream=False,
                max_tokens=2000,
            )

            ai_response = response.choices[0].message.content
            if "registered workers" in ai_response:
                self.current_assignment = "registered"
                print("意图识别:",ai_response)
                print("switch to <registered>")
                self.messages[self.current_assignment].append({"role": "user", "content": user_input})
            elif "query workers" in ai_response:
                self.current_assignment = "query"
                print("意图识别:",ai_response)
                print("switch to <query>")
                self.messages[self.current_assignment].append({"role": "user", "content": user_input})
            elif "delete workers" in ai_response:
                self.current_assignment = "delete"
                print("意图识别:",ai_response)
                print("switch to <delete>")
                self.messages[self.current_assignment].append({"role": "user", "content": user_input})
            elif "customer service" in ai_response:
                print("意图识别:",ai_response)
                print("switch to <customer service>")
                self.messages["system"] += self.messages[self.current_assignment]
                self.current_assignment = "system"
                return ai_response
            else:
                self.messages[self.current_assignment].append({"role": "assistant", "content": ai_response})
                return ai_response

    def start_conversation(self):
        while True:
            user_input = input("User: ")
            print("\n===================================\n", "User: ", user_input)
            if user_input.lower() in ['exit', 'quit']:
                print("Exiting conversation.")
                break
            response = self.get_response(user_input)
            print("Assistant:", response)

In [21]:
assistant = SmartAssistant()
assistant.start_conversation()


 User:  你好啊，我是二郎神
Assistant: 您好！我是客服助手，很高兴为您服务。如果您需要帮助，例如用户注册、查询用户数据或删除用户数据，请告诉我具体需求，我会为您调用相应的处理程序。请问您需要什么帮助？

 User:  怎么进行注册
意图识别: 好的，您需要进行用户注册。我将调用“registered workers”来处理您的请求。请稍等，我们会尽快为您服务。
switch to <registered>
Assistant: 欢迎！为了完成注册，请提供以下信息：

1. **用户名、性别和年龄**  
   - 例如：张三，男，25岁  

2. **密码**  
   - 请设置一个安全的密码  

3. **电子邮件地址**  
   - 例如：zhangsan@example.com  

如果您已准备好这些信息，请直接提供；如果缺少某项，我会提示您补充。注册成功后，我会生成一个用户ID并返回给您。

 User:  你知道我是谁？
Assistant: 目前我无法识别您的具体身份，因为您还没有提供注册所需的个人信息（如用户名、性别、年龄、密码和电子邮件地址）。如果您想注册，请提供这些信息，我会为您生成用户ID并完成存储。如果需要帮助，请随时告知！

 User:  好的，张三，男，100岁，
Assistant: 感谢提供信息！以下是您已提交的内容：  
- **用户名**：张三  
- **性别**：男  
- **年龄**：100岁  

还缺少以下必要信息才能完成注册：  
1. **密码**：请设置您的账户密码。  
2. **电子邮件地址**：请提供有效的电子邮箱。  

请补充这些信息，我会立即为您完成注册并返回用户ID。

 User:  密码是110，邮件是123@2333.com
意图识别: 感谢您提供完整信息！以下是您的注册详情：  
- **用户名**：张三  
- **性别**：男  
- **年龄**：100岁  
- **密码**：110（请注意密码安全性）  
- **电子邮件**：123@2333.com  

以下是生成的用户ID和数据库操作：  
```sql
INSERT INTO users (user_id, username, gender, age, password, email) 
V